In [0]:
dbutils.widgets.dropdown("catalog_name", "workspace", ["workspace", "dbr_dev"], "Target Catalog")
dbutils.widgets.text("bronze_schema", "joshuandegwa_bronze", "Bronze Schema")
dbutils.widgets.text("silver_schema", "joshuandegwa_silver", "Silver Schema")
dbutils.widgets.text("gold_schema", "joshuandegwa_gold", "Gold Schema")

# Retrieve the values from the widgets
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("bronze_schema")
table_name = "bronze_opensky_flights"
target_table_fqn = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"

In [0]:
import requests
from pyspark.sql.functions import current_timestamp, current_date
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, ArrayType, BooleanType
)

# Target Definitions
# catalog_name = "workspace"
# schema_name = "joshuandegwa_bronze"
# table_name = "bronze_opensky_flights"
# target_table_fqn = f"`{catalog_name}`.`{schema_name}`.`{table_name}`"

# 1. Fetch Data from OpenSky REST API
url = "https://opensky-network.org/api/states/all"
response = requests.get(url, timeout=30)
response.raise_for_status()

payload = response.json()
fetched_time = payload.get("time")
raw_states = payload.get("states", [])

print(f"Successfully fetched {len(raw_states)} flight records for timestamp: {fetched_time}")

# Handle empty state returns gracefully
if not raw_states:
    print("No flight states returned by API.")
else:
    # 2. Define Schema matching OpenSky state vector elements
    # Reference: https://openskynetwork.github.io/opensky-api/rest.html#all-state-vectors
    schema = StructType([
        StructField("icao24", StringType(), True),
        StructField("callsign", StringType(), True),
        StructField("origin_country", StringType(), True),
        StructField("time_position", IntegerType(), True),
        StructField("last_contact", IntegerType(), True),
        StructField("longitude", DoubleType(), True),
        StructField("latitude", DoubleType(), True),
        StructField("baro_altitude", DoubleType(), True),
        StructField("on_ground", BooleanType(), True),
        StructField("velocity", DoubleType(), True),
        StructField("true_track", DoubleType(), True),
        StructField("vertical_rate", DoubleType(), True),
        StructField("sensors", ArrayType(IntegerType()), True),
        StructField("geo_altitude", DoubleType(), True),
        StructField("squawk", StringType(), True),
        StructField("spi", BooleanType(), True),
        StructField("position_source", IntegerType(), True)
    ])

    # 3. Create Spark DataFrame
    df = spark.createDataFrame(raw_states, schema)

    # 4. Add Audit Columns and Append to Delta Table
    df_enriched = (
        df.withColumn("_api_snapshot_time", current_timestamp())
          .withColumn("_load_date", current_date())
    )

    (
        df_enriched.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table_fqn)
    )

    print(f"Successfully appended batch to {target_table_fqn}")

Successfully fetched 9498 flight records for timestamp: 1790723830
Successfully appended batch to `workspace`.`joshuandegwa_bronze`.`bronze_opensky_flights`


In [0]:
record_count = spark.read.table(f"{catalog_name}.{schema_name}.bronze_opensky_flights").count()
print(f"Total records: {record_count}")

Total records: 19072
